### Giá Trị Kỳ Dị SVD (Lớn Nhất) - Có Xuống Thàng


In [4]:
import numpy as np
from IPython.display import display, Math, Markdown

def SVD_Reduced(A_input, num_components=None, v0_input=None, num_iters=1000, tol=1e-10):
    def matrix_to_latex(M, precision=4):
        if not isinstance(M, np.ndarray): return str(M)
        elif M.ndim == 1:
            inner = " \\\\ ".join([f"{x:.{precision}f}" for x in M])
            return f"\\begin{{bmatrix}} {inner} \\end{{bmatrix}}^T"
        else:
            rows = " \\\\ ".join([" & ".join([f"{x:.{precision}f}" for x in row]) for row in M])
            return f"\\begin{{bmatrix}} {rows} \\end{{bmatrix}}"

    def power_method_L2(B, v0, num_iters, tol):
        v = v0.copy()
        lam_prev = 0
        for i in range(num_iters):
            y_new = B @ v
            lam_rayleigh = v.T @ y_new
            norm_y = np.linalg.norm(y_new, 2)
            if norm_y == 0: return 0, v
            v_new = y_new / norm_y
            
            idx = np.argmax(np.abs(v_new))
            if v_new[idx] < 0: v_new = -v_new
                
            if np.abs(lam_rayleigh - lam_prev) < tol:
                v = v_new
                lam_prev = lam_rayleigh
                break
            v = v_new
            lam_prev = lam_rayleigh
        return lam_prev, v

    # Phần 1: Ghi chú lý thuyết Phương pháp Xuống thàng
    theory_md = [
        "## ❖ KHAI TRIỂN KỲ DỊ SVD BẰNG PHƯƠNG PHÁP XUỐNG THANG (DEFLATION)",
        "",
        "**Ý tưởng của phương pháp Xuống thàng:**",
        "Sau khi đã tìm được trị riêng trội lớn nhất $\\lambda_1$ và véctơ riêng tương ứng $v_1$ của ma trận $A$. "
        "Để tìm được trị riêng lớn thứ hai, ta cần 'khử' (triệt tiêu) ảnh hưởng của $\\lambda_1$ bằng cách tạo ra một ma trận mới:",
        "$$ A_1 = A - \\lambda_1 v_1 x^T $$",
        "(với $x$ là véctơ sao cho $x^T v_1 = 1$). Mọi trị riêng của $A_1$ đều giống $A$, ngoại trừ $\\lambda_1$ đã bị ép về 0. "
        "Do đó, khi áp dụng tiếp phương pháp Lũy thừa lên $A_1$, thuật toán sẽ hội tụ về trị riêng lớn thứ hai $\\lambda_2$ của $A$. "
        "Quá trình này lặp lại để tìm các trị riêng tiếp theo."
    ]
    display(Markdown('\n'.join(theory_md)))
    
    A = np.array(A_input, dtype=float)
    n, m = A.shape
    
    display(Math(f"A = {matrix_to_latex(A, precision=4)}"))

    A_k = A.copy()
    rank = min(n, m)
    if num_components is not None:
        rank = min(rank, num_components)
    
    if v0_input is None: v0_base = np.ones(m) / np.sqrt(m)
    else: v0_base = np.array(v0_input, dtype=float).flatten()
    
    U_cols = []
    S_vals = []
    V_cols = []
    
    approx_formula_terms = []
    
    for k in range(1, rank + 1):
        display(Markdown(f"\n### --- BƯỚC {k} ---"))
        B_k = A_k.T @ A_k
        lam, v = power_method_L2(B_k, v0_base, num_iters, tol)
        
        if lam < 1e-10:
            display(Markdown(f"⚠️ **Dừng thuật toán:** Giá trị riêng cực đại tiếp theo xấp xỉ 0."))
            break
            
        sigma = np.sqrt(lam)
        u = (A_k @ v) / sigma
        
        U_cols.append(u)
        S_vals.append(sigma)
        V_cols.append(v)
        
        display(Markdown(f"- **Giá trị kỳ dị lớn nhất (hiện tại):** $\\sigma_{k} = \\sqrt{{\\lambda_{k}}} \\approx {sigma:.4f}$"))
        display(Markdown(f"- **Vector kỳ dị phải:**"))
        display(Math(f"v_{k} = {matrix_to_latex(v, precision=4)}"))
        display(Markdown(f"- **Vector kỳ dị trái:**"))
        display(Math(f"u_{k} = \\frac{{A_{k} v_{k}}}{{\\sigma_{k}}} = {matrix_to_latex(u, precision=4)}"))
        
        approx_formula_terms.append(f"\\sigma_{k} u_{k} v_{k}^T")
        
        A_next = A_k - sigma * np.outer(u, v)
        A_k = A_next
        
    display(Markdown("\n### ❖ TỔNG KẾT VÀ XẤP XỈ MA TRẬN A"))
    approx_formula = "A \\approx " + " + ".join(approx_formula_terms)
    display(Math(approx_formula))

# DỮ LIỆU ĐỀ BÀI

# ═══════════════════════════════════════════════════════════════════
# NHẬP DỮ LIỆU CỦA BẠN VÀO ĐÂY
# ═══════════════════════════════════════════════════════════════════
A = np.array([
    [3.0, 2.0, 1.0],
    [1.0, 4.0, 2.0],
    [2.0, 1.0, 5.0]
], dtype=float)
# ═══════════════════════════════════════════════════════════════════

# num_components: so luong gia tri ky di can tinh (None = tinh het)
SVD_Reduced(A, num_components=2)


## ❖ KHAI TRIỂN KỲ DỊ SVD BẰNG PHƯƠNG PHÁP XUỐNG THANG (DEFLATION)

**Ý tưởng của phương pháp Xuống thàng:**
Sau khi đã tìm được trị riêng trội lớn nhất $\lambda_1$ và véctơ riêng tương ứng $v_1$ của ma trận $A$. Để tìm được trị riêng lớn thứ hai, ta cần 'khử' (triệt tiêu) ảnh hưởng của $\lambda_1$ bằng cách tạo ra một ma trận mới:
$$ A_1 = A - \lambda_1 v_1 x^T $$
(với $x$ là véctơ sao cho $x^T v_1 = 1$). Mọi trị riêng của $A_1$ đều giống $A$, ngoại trừ $\lambda_1$ đã bị ép về 0. Do đó, khi áp dụng tiếp phương pháp Lũy thừa lên $A_1$, thuật toán sẽ hội tụ về trị riêng lớn thứ hai $\lambda_2$ của $A$. Quá trình này lặp lại để tìm các trị riêng tiếp theo.

<IPython.core.display.Math object>


### --- BƯỚC 1 ---

- **Giá trị kỳ dị lớn nhất (hiện tại):** $\sigma_1 = \sqrt{\lambda_1} \approx 7.1483$

- **Vector kỳ dị phải:**

...

<IPython.core.display.Math object>

- **Vector kỳ dị trái:**

<IPython.core.display.Math object>


### ❖ TỔNG KẾT VÀ XẤP XỈ MA TRẬN A

<IPython.core.display.Math object>